# Week 1 - Applicant Data Cleaning
By Shakti | NextGenLearners Internship

In [24]:
import pandas as pd


In [25]:
df = pd.read_csv('applicants.csv')
df.head()

,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status
0,Steven Miller,jessebenson@example.net,0317-1484348,digital marketing,FAST NUCES,unknown,rejected
1,Megan Joyce,brownjessica@example.org,0312-7194416,appdev,FAST NUCES,15/10/2025,under review
2,Danielle Odonnell,DAVISRODNEY@EXAMPLE.COM,0334-8030010,DIGITAL MKT,Karachi University,01/08/2025,NaN
3,Thomas Bradley,jason76@example.net,0310-6438436,WEB DEV,LUMS,30/12/2025,Under Review
4,Kimberly Smith,housemelinda@example.org,0320-2813543,webdev,Karachi University,11/12/2025,SELECTED


In [26]:
df.shape

(95, 7)

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Applicant Name    92 non-null     object
 1   Email             87 non-null     object
 2   Phone             95 non-null     object
 3   Domain Applied    95 non-null     object
 4   University        82 non-null     object
 5   Application Date  95 non-null     object
 6   Status            91 non-null     object
dtypes: object(7)
memory usage: 5.3+ KB


In [28]:
df.isnull().sum()

,0
Applicant Name,3
Email,8
Phone,0
Domain Applied,0
University,13
Application Date,0
Status,4


In [29]:
df.duplicated().sum()

np.int64(3)

## Before Cleaning - Observations

- The dataset has 95 rows and 7 columns
- Missing values found: Applicant Name (3), Email (8), University (13), Status (4)
- 3 exact duplicate rows were found
- Application Date column is currently stored as text (object type), not as an actual date

In [30]:
# Applicant Name missing -> row hi drop kar do (naam ke bina application useless hai)
df = df.dropna(subset=['Applicant Name'])

In [31]:
# Email missing -> row rakhni hai, placeholder daal do
df['Email'] = df['Email'].fillna('not_provided@example.com')

In [32]:
# University missing -> "Not Specified" se fill karo (critical nahi hai)
df['University'] = df['University'].fillna('Not Specified')

In [33]:
# Status missing -> default "Under Review" (matlab abhi process nahi hui)
df['Status'] = df['Status'].fillna('Under Review')

In [34]:
df.isnull().sum()

,0
Applicant Name,0
Email,0
Phone,0
Domain Applied,0
University,0
Application Date,0
Status,0


## Missing Value Handling - Reasoning

- Applicant Name: Rows with missing names were dropped, since a nameless application
  is not usable for identification or outreach.
- Email: Missing values were filled with a placeholder ("not_provided@example.com")
  instead of dropping, to retain the row for further analysis.
- University: Filled with "Not Specified" since this field is not critical for
  determining application status.
- Status: Filled with "Under Review" as the default, since a missing status likely
  means the application has not been processed yet.

In [35]:
# Check exact duplicates first
df.duplicated().sum()

np.int64(3)

In [36]:
# Remove exact duplicate rows
df = df.drop_duplicates()

In [37]:
# Check duplicates by Email too (catches same person with slightly different name formatting)
df = df.drop_duplicates(subset=['Email'], keep='first')

In [38]:
df.shape

(81, 7)

## Duplicate Removal

Exact duplicate rows were removed using drop_duplicates(). Additionally, duplicates
were checked based on the Email column (keeping the first occurrence), since Email is
a more reliable unique identifier than the full row or Name — it catches cases where
the same person applied twice with slightly different formatting.

In [39]:
# Extra spaces hatao naam aur email se
df['Applicant Name'] = df['Applicant Name'].str.strip()
df['Email'] = df['Email'].str.strip().str.lower()

In [40]:
df['Domain Applied'].unique()

array(['digital marketing', 'appdev', 'DIGITAL MKT', 'WEB DEV', 'webdev',
       'app dev', 'GRAPHIC DESIGN', 'Data Science', 'data science',
       'graphic design', 'web dev', 'Graphic Design', 'DataScience',
       'DATA SCIENCE', 'APP DEV', 'Web development', 'Digital Marketing',
       'App Development', 'Web Development'], dtype=object)

In [41]:
domain_mapping = {
    'web dev': 'Web Development',
    'WEB DEV': 'Web Development',
    'webdev': 'Web Development',
    'Web development': 'Web Development',
    'Web Development': 'Web Development',

    'app dev': 'App Development',
    'APP DEV': 'App Development',
    'appdev': 'App Development',
    'App Development': 'App Development',

    'data science': 'Data Science',
    'DATA SCIENCE': 'Data Science',
    'DataScience': 'Data Science',
    'Data Science': 'Data Science',

    'graphic design': 'Graphic Design',
    'GRAPHIC DESIGN': 'Graphic Design',
    'Graphic Design': 'Graphic Design',

    'digital marketing': 'Digital Marketing',
    'DIGITAL MKT': 'Digital Marketing',
    'Digital Marketing': 'Digital Marketing',
}

df['Domain Applied'] = df['Domain Applied'].map(domain_mapping)
df['Domain Applied'].unique()

array(['Digital Marketing', 'App Development', 'Web Development',
       'Graphic Design', 'Data Science'], dtype=object)

In [42]:
df['Status'] = df['Status'].str.strip().str.title()
df['Status'].unique()

array(['Rejected', 'Under Review', 'Selected'], dtype=object)

## Text Standardization

Whitespace was stripped from Applicant Name and Email fields, and Email was
lowercased for consistency. The Domain Applied column had 19 different spelling
and casing variations for only 5 real domains — these were mapped to 5 standard
categories using a mapping dictionary. The Status column was standardized to
Title Case (e.g., "Under Review", "Selected", "Rejected").

In [43]:
df['Application Date'] = pd.to_datetime(df['Application Date'], errors='coerce')
df['Application Date'].isnull().sum()

/tmp/ipykernel_3537/1367030752.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Application Date'] = pd.to_datetime(df['Application Date'], errors='coerce')


np.int64(4)

In [44]:
df[df['Application Date'].isnull()]

,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status
0,Steven Miller,jessebenson@example.net,0317-1484348,Digital Marketing,FAST NUCES,NaT,Rejected
73,Spencer Johnston,wsmith@example.com,0318 2171218,Graphic Design,Not Specified,NaT,Selected
91,Kimberly Webb,lpetersen@example.com,0324-7667458,Digital Marketing,NED University,NaT,Rejected
94,Jerry Henderson,steven17@example.net,0336-3592974,App Development,FAST NUCES,NaT,Selected


In [45]:
df['Phone'] = df['Phone'].astype(str).str.replace('-', '', regex=False).str.replace(' ', '', regex=False)
df['Phone'].head()

,Phone
0,03171484348
1,03127194416
2,03348030010
3,03106438436
4,03202813543


In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 81 entries, 0 to 94
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Applicant Name    81 non-null     object        
 1   Email             81 non-null     object        
 2   Phone             81 non-null     object        
 3   Domain Applied    81 non-null     object        
 4   University        81 non-null     object        
 5   Application Date  77 non-null     datetime64[ns]
 6   Status            81 non-null     object        
dtypes: datetime64[ns](1), object(6)
memory usage: 5.1+ KB


## Data Type Fixes

Application Date was converted from text to a proper datetime type using
pd.to_datetime() with errors='coerce', which converts unreadable date formats
into NaT (missing) instead of crashing. 4 rows had unparseable dates and were
kept as NaT. Phone numbers were cleaned by removing dashes and spaces for
consistent formatting, while keeping them as text since phone numbers with
leading zeros shouldn't be converted to numeric type.

## Data Quality Summary

**Starting dataset:** 95 rows, 7 columns

**Issues found:**
- 3 missing Applicant Name, 8 missing Email, 13 missing University, 4 missing Status
- 3 exact duplicate rows, plus additional duplicates identified by matching Email
- Domain Applied had 19 different spelling/casing variations for only 5 real domains
- Application Date was stored as text, not as an actual date type
- Phone numbers had inconsistent dash/space formatting
- 6 rows had unparseable dates (later reduced to 4 after duplicate removal)

**Actions taken:**
- Dropped rows with missing Applicant Name
- Filled missing Email with a placeholder
- Filled missing University with "Not Specified"
- Filled missing Status with "Under Review"
- Removed exact and email-based duplicate rows
- Standardized all Domain Applied values into 5 consistent categories
- Standardized Status into Title Case
- Converted Application Date to proper datetime format
- Cleaned Phone number formatting for consistency

**Final dataset:** 81 rows, 7 columns, no missing values in critical fields,
no duplicates, consistent formatting throughout.

In [48]:
df.to_csv('applicants_cleaned.csv', index=False)